# Type-aware bits + token-weighted G²PTQ on Colab

Runtime: **A100 GPU** (Runtime → Change runtime type), High-RAM if offered.

Cells 1–5 set everything up (about 20–40 min, mostly compiling CUDA extensions). After that, run the
experiment steps in order. Every step skips work that is already finished, so you can re-run a cell.

Two Python environments are used: the system Python for quantization (pinned torch from
`requirements.txt`), and `/content/venv-eval` for vLLM (IFEval and the self-generated data).
The scripts pick the right one through `PY_EVAL`.

Don't `import torch` in this notebook: the install replaces Colab's torch, and the scripts run
in their own processes.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## 2. Settings

In [ ]:
import os

REPO_URL = "https://github.com/Astatium5/G2PTQ.git"  # private repo: https://<token>@github.com/...
REPO_DIR = "/content/G2PTQ"
os.environ["PY_EVAL"] = "/content/venv-eval/bin/python"

try:  # optional: add HF_TOKEN under Colab "Secrets" to avoid Hub rate limits
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("No HF_TOKEN secret (fine: the model and datasets are public)")

## 3. Get the code

In [ ]:
import os
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull && git -C {REPO_DIR} submodule update --init --recursive
else:
    !git clone --recursive {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -3

## 4. Quantization environment (system Python)

In [ ]:
%%bash
set -e
cd /content/G2PTQ
pip install -q uv packaging ninja
uv pip install --system -r requirements.txt --torch-backend=auto
# Required CUDA extensions (A100: skip FlashAttention-3, it is Hopper-only)
MAX_JOBS=8 python -m pip install -q --no-build-isolation ./third-party/fast-hadamard-transform
CAUSAL_CONV1D_FORCE_BUILD=TRUE MAX_JOBS=8 python -m pip install -q --no-build-isolation ./third-party/causal-conv1d
python -c "import torch, transformers, fla, causal_conv1d, fast_hadamard_transform; \
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name())"

## 5. Evaluation environment (vLLM)

In [ ]:
%%bash
set -e
cd /content/G2PTQ
uv venv /content/venv-eval --python 3.12
uv pip install --python /content/venv-eval/bin/python vllm -r ours/requirements-eval.txt
/content/venv-eval/bin/python -c "import vllm, lm_eval; print('vllm', vllm.__version__)"

## 6. Quick CPU tests (optional, 1 min)
These check our code paths without the GPU.

In [ ]:
%%bash
cd /content/G2PTQ
python tests/test_bits_map.py && python tests/test_token_weights.py && python tests/test_token_weight_signals.py

## Step 0: prepare
Downloads Qwen3.5-4B and wikitext, builds the calibration sets (decontaminated against IFEval/IFBench),
generates the self-generated set with vLLM, and prints the size options.

In [ ]:
%%bash
cd /content/G2PTQ
bash scripts/ours/00_prepare.sh 2>&1 | tee -a outputs_00_prepare.log

## Step 1: check run (go/no-go)
Small G²PTQ run on 4B with 16 sequences, then IFEval on 50 prompts. Look at the time and peak GPU memory
at the end of the output: they tell you how long the full grid will take.

In [ ]:
%%bash
cd /content/G2PTQ
bash scripts/ours/01_check_run.sh 2>&1 | tee -a outputs_01_check.log

## Change settings
Edits `scripts/ours/config.sh`. Use it after step 2 (uniform baseline size) and step 3a (λ, mode).

In [ ]:
import re, pathlib

def set_cfg(**values):
    cfg = pathlib.Path(REPO_DIR) / "scripts/ours/config.sh"
    text = cfg.read_text()
    for key, val in values.items():
        text, n = re.subn(rf"^{key}=.*$", f"{key}={val}", text, flags=re.M)
        assert n == 1, f"{key} not found in config.sh"
    cfg.write_text(text)
    print("\n".join(l for l in text.splitlines() if l.split("=")[0] in values))

# Examples (uncomment as needed):
# set_cfg(UNIFORM_BITS=3, UNIFORM_GS=32)
# set_cfg(LAMBDA=1.0, TW_MODE="source")

## Step 2: bit search (Idea 1)
Sensitivity per layer type, search under the budget, GPTQ confirmation →
`outputs/bit_search/best_bits_map.json`. Then size-match the uniform baseline (next cell) and set it with `set_cfg`.

In [ ]:
%%bash
cd /content/G2PTQ
bash scripts/ours/02_bit_search.sh 2>&1 | tee -a outputs_02_bit_search.log

In [ ]:
%%bash
cd /content/G2PTQ
M=./modelzoo/Qwen3.5/Qwen3.5-4B
python ours/size/bpw.py --model $M --w_groupsize 128 --w_bits_map outputs/bit_search/best_bits_map.json
BPW=$(python ours/size/bpw.py --model $M --w_groupsize 128 --w_bits_map outputs/bit_search/best_bits_map.json --json \
      | python -c "import json,sys; print(round(json.load(sys.stdin)['avg_bpw_quantized'], 3))")
python ours/size/bpw.py --model $M --match $BPW

## Step 3: token weights (Idea 2)
3a tunes λ and the weighting mode on held-out KL (8 G²PTQ runs; skip it if short on time and keep λ=1, source).
3b computes the weights for every calibration set.

In [ ]:
%%bash
cd /content/G2PTQ
bash scripts/ours/03_token_weights.sh --sweep 2>&1 | tee -a outputs_03a_sweep.log

In [ ]:
%%bash
cd /content/G2PTQ
bash scripts/ours/03_token_weights.sh 2>&1 | tee -a outputs_03b_weights.log

## Step 4: main grid
12 G²PTQ + 6 GPTQ runs. To split between two Colab sessions, pass calibration sets, e.g.
`bash scripts/ours/04_grid.sh wikitext2 calib_ifchat` in one and `bash scripts/ours/04_grid.sh calib_selfgen` in the other.

In [ ]:
%%bash
cd /content/G2PTQ
bash scripts/ours/04_grid.sh 2>&1 | tee -a outputs_04_grid.log

## Step 5: IFEval, results table, significance tests

In [ ]:
%%bash
cd /content/G2PTQ
bash scripts/ours/05_eval.sh 2>&1 | tee -a outputs_05_eval.log

In [ ]:
import pandas as pd
df = pd.read_csv(f"{REPO_DIR}/outputs/results.csv")
cols = [c for c in ["exp", "method", "calib", "avg_bpw", "KL-calib_heldout_if", "KL-wikitext2",
                    "ifeval_prompt_strict", "ifeval_inst_strict", "quant_hours", "peak_gpu_gb"] if c in df]
df[cols].sort_values("ifeval_prompt_strict", ascending=False) if "ifeval_prompt_strict" in df else df[cols]

## Save results to Google Drive
Copies metrics, IFEval outputs, bootstrap results and logs (not the model checkpoints). Run it whenever you
want a backup; `/content` is wiped when the runtime ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/g2ptq_results
!rsync -a --exclude 'dequant_model' --exclude 'export_model' {REPO_DIR}/outputs/ /content/drive/MyDrive/g2ptq_results/outputs/
!cp {REPO_DIR}/outputs_*.log /content/drive/MyDrive/g2ptq_results/ 2>/dev/null; ls /content/drive/MyDrive/g2ptq_results